# 5.5 · 接入真实宿主：本地与远程服务器

**Connecting real hosts: local and remote servers**

> **学完本课你能：** 配置 Claude Desktop 及同类宿主，使其启动本地 stdio 服务器并连接远程服务器，并能从日志中诊断各种典型的配置故障。

官方阅读材料：[Connect to local MCP servers](https://modelcontextprotocol.io/docs/2026-07-28/develop/connect-local-servers) · [Connect to remote MCP servers](https://modelcontextprotocol.io/docs/2026-07-28/develop/connect-remote-servers)

从上到下依次运行单元格（`Shift+Enter`），然后改动代码再重新运行。所有内容都在本地运行：服务器位于当前内核、临时目录或 localhost 上。调用真实大语言模型（LLM）的单元格只有在设置了 `ANTHROPIC_API_KEY` 时才会执行，否则自动跳过。

### 1. 一个行为类似文件系统服务器的本地服务器
官方的文件系统服务器（Filesystem Server）把它可以访问的目录作为<strong>命令行参数</strong>接收。我们的服务器也这样做，而且和许多真实服务器一样，它还需要从<strong>环境变量（Environment Variable）</strong> 中读取一个密钥。它把启动信息写到<strong>标准错误（stderr）</strong>，而宿主的逐服务器日志正是来自这里。所有内容都放在一个临时文件夹中。

In [ ]:
import asyncio
import json
import logging
import os
import platform
import shutil
import socket
import sys
import tempfile
import textwrap
import threading
import time
from contextlib import AsyncExitStack
from pathlib import Path

from mcp import Client, StdioServerParameters
from mcp.client.stdio import stdio_client

for name in ("httpx", "httpx2", "mcp", "uvicorn"):
    logging.getLogger(name).setLevel(logging.WARNING)

work = Path(tempfile.mkdtemp())
(work / "Desktop").mkdir()
(work / "Downloads").mkdir()
(work / "Desktop" / "todo.md").write_text("- learn MCP\n- write a server\n")
(work / "Downloads" / "invoice.txt").write_text("Invoice #42: $120\n")

SERVER = work / "notes_server.py"
SERVER.write_text(textwrap.dedent('''
    import os
    import sys
    from pathlib import Path

    from mcp.server import MCPServer
    from mcp.server.mcpserver.exceptions import ToolError

    TOKEN = os.environ.get("NOTES_API_TOKEN")
    if not TOKEN:
        print("notes: NOTES_API_TOKEN is not set; refusing to start", file=sys.stderr)
        sys.exit(1)
    ROOTS = [Path(arg).resolve() for arg in sys.argv[1:]]
    for root in ROOTS:
        if not root.is_dir():
            print(f"notes: allowed directory {root} does not exist (cwd={os.getcwd()})", file=sys.stderr)
            sys.exit(1)
    print(f"notes: serving {len(ROOTS)} directories", file=sys.stderr)

    server = MCPServer("notes")


    def inside(path: str) -> Path:
        p = Path(path).resolve()
        if not any(p.is_relative_to(r) for r in ROOTS):
            raise ToolError(f"{path} is outside the allowed directories")
        return p


    @server.tool()
    def list_files() -> list[str]:
        """List every file in the allowed directories (absolute paths)."""
        return sorted(str(f) for r in ROOTS for f in r.rglob("*") if f.is_file())


    @server.tool()
    def read_file(path: str) -> str:
        """Read a text file inside an allowed directory."""
        return inside(path).read_text()


    @server.tool()
    def write_file(path: str, text: str) -> str:
        """Create or overwrite a text file inside an allowed directory."""
        inside(path).write_text(text)
        return f"wrote {len(text)} chars to {path}"


    @server.tool()
    def visible_env() -> list[str]:
        """Names (never values) of the environment variables this server process can see."""
        return sorted(os.environ)


    if __name__ == "__main__":
        server.run(transport="stdio")
'''))
print("server written to", SERVER)

### 2. 编写并校验一个 `claude_desktop_config.json`
这正是 Claude Desktop 读取的结构：顶层是一个 `mcpServers` 对象，它的键是界面中显示的友好名称，值则说明如何<strong>启动</strong>服务器（`command`、`args`、可选的 `env`）。根据文档，真实位置在 macOS 上是 `~/Library/Application Support/Claude/claude_desktop_config.json`，在 Windows 上是 `%APPDATA%\Claude\claude_desktop_config.json`。我们则把自己的配置写到临时文件夹中。

In [ ]:
config = {
    "mcpServers": {
        "notes": {
            "command": sys.executable,                      # absolute path to the interpreter
            "args": [str(SERVER), str(work / "Desktop"), str(work / "Downloads")],
            "env": {"NOTES_API_TOKEN": "demo-token-1234"},
        }
    }
}
CONFIG_PATH = work / "claude_desktop_config.json"
CONFIG_PATH.write_text(json.dumps(config, indent=2))

real_location = {"Darwin": "~/Library/Application Support/Claude/claude_desktop_config.json",
                 "Windows": r"%APPDATA%\Claude\claude_desktop_config.json"}
print("on this OS Claude Desktop would read:",
      real_location.get(platform.system(), "(not listed in the docs for this OS; use Settings → Developer → Edit Config)"))
print(CONFIG_PATH.read_text())

服务器根本没启动时，宿主（Host）能告诉你的信息很少，所以要在重启应用之前先检查配置。这个校验器把文档中的故障排查清单编码成了检查项：合法的 JSON、`mcpServers` 键、绝对路径、存在的命令，以及只含字符串值的 `env`。

In [ ]:
def validate_config(text: str) -> list[str]:
    try:
        data = json.loads(text)
    except json.JSONDecodeError as exc:
        return [f"invalid JSON at line {exc.lineno}, column {exc.colno}: {exc.msg}"]
    servers = data.get("mcpServers")
    if not isinstance(servers, dict):
        return ["top-level `mcpServers` object is missing"]
    problems = []
    for name, entry in servers.items():
        cmd = entry.get("command")
        if not isinstance(cmd, str) or not cmd:
            problems.append(f"{name}: `command` must be a non-empty string")
        elif not (Path(cmd).is_absolute() and Path(cmd).exists()) and shutil.which(cmd) is None:
            problems.append(f"{name}: command {cmd!r} not found on PATH (try an absolute path)")
        args = entry.get("args", [])
        if not isinstance(args, list) or not all(isinstance(a, str) for a in args):
            problems.append(f"{name}: `args` must be a list of strings")
        else:
            for a in args:
                looks_like_path = "/" in a or "\\" in a or a.startswith(".")
                if looks_like_path and not Path(a).is_absolute():
                    problems.append(f"{name}: relative path {a!r} (the host's working directory is undefined)")
        env = entry.get("env", {})
        if not isinstance(env, dict) or not all(isinstance(v, str) for v in env.values()):
            problems.append(f"{name}: `env` must map names to string values")
    return problems


print("our config:", validate_config(CONFIG_PATH.read_text()) or "OK")
broken = '''{
  "mcpServers": {
    "notes": {"command": "pyhton3", "args": ["./notes_server.py", "Desktop"], "env": {"PORT": 8080}},
  }
}'''
print("broken (trailing comma):", validate_config(broken))
print("broken (comma fixed):  ", *validate_config(broken.replace("},\n  }", "}\n  }")), sep="\n  ")

### 3. 扮演宿主：读取配置、启动所有服务器、列出工具
这就是 Claude Desktop 启动时所做的事：读取 `mcpServers`，用 `command`、`args` 和 `env` 把每个服务器作为子进程启动，并通过 stdio 与之通信。和真实应用一样，我们的宿主（Host）为<strong>每个服务器写一个日志文件</strong>，记录该服务器的 stderr（`mcp-server-NAME.log`），另外再写一个通用的 `mcp.log`。

In [ ]:
LOGS = work / "logs"
LOGS.mkdir(exist_ok=True)


def host_log(msg: str) -> None:
    with open(LOGS / "mcp.log", "a") as f:
        f.write(f"{time.strftime('%H:%M:%S')} {msg}\n")


def params_for(entry: dict) -> StdioServerParameters:
    return StdioServerParameters(command=entry["command"], args=entry.get("args", []),
                                 env=entry.get("env"), cwd=entry.get("cwd"))


async def connect_all(stack: AsyncExitStack, cfg: dict) -> dict[str, Client]:
    clients = {}
    for name, entry in cfg["mcpServers"].items():
        errlog = stack.enter_context(open(LOGS / f"mcp-server-{name}.log", "a"))
        host_log(f"[{name}] launching: {entry['command']} {' '.join(entry.get('args', []))}")
        try:
            clients[name] = await stack.enter_async_context(Client(stdio_client(params_for(entry), errlog=errlog)))
            host_log(f"[{name}] connected, protocol {clients[name].protocol_version}")
        except Exception as exc:
            host_log(f"[{name}] FAILED: {exc!r}")
    return clients


cfg = json.loads(CONFIG_PATH.read_text())
async with AsyncExitStack() as stack:
    clients = await connect_all(stack, cfg)
    for name, client in clients.items():
        tools = (await client.list_tools()).tools
        print(f"{name}: {[t.name for t in tools]}")
    notes = clients["notes"]
    files = (await notes.call_tool("list_files", {})).structured_content["result"]
    print("list_files ->", [Path(f).name for f in files])
    print("read_file  ->", (await notes.call_tool("read_file", {"path": files[0]})).content[0].text.strip())
    escape = await notes.call_tool("read_file", {"path": str(SERVER)})
    print("outside the allowed dirs ->", escape.is_error, escape.content[0].text)

print("\n--- mcp.log ---\n" + (LOGS / "mcp.log").read_text())
print("--- mcp-server-notes.log ---\n" + (LOGS / "mcp-server-notes.log").read_text())

### 4. stdio 服务器实际拿到的是什么环境？
调试指南警告说，stdio 服务器<strong>只继承有限的一部分环境变量</strong>（Environment Variable Inheritance）。Python SDK 把这一点具体化了：它传递一个小小的白名单（POSIX 上为 `HOME`、`LOGNAME`、`PATH`、`SHELL`、`TERM`、`USER`），再与你的 `env` 合并。在 shell 中导出的密钥<strong>不会</strong>传到服务器，除非你把它写进 `env`。

In [ ]:
os.environ["MY_SHELL_SECRET"] = "only-in-the-host"
async with Client(params_for(cfg["mcpServers"]["notes"])) as c:
    seen = (await c.call_tool("visible_env", {})).structured_content["result"]
print("server sees:", seen)
print("MY_SHELL_SECRET visible?", "MY_SHELL_SECRET" in seen, "| NOTES_API_TOKEN visible?", "NOTES_API_TOKEN" in seen)

### 5. 审批如何运作：宿主中的同意关卡
在 Claude Desktop 中，<em>每一个</em>文件系统操作都要等待你的批准。这道关卡属于<strong>宿主（Host）</strong>，而不属于协议：在转发 `tools/call` 之前，宿主会询问用户。这里由一个脚本化的“用户”批准读操作、拒绝写操作。被拒绝的调用永远不会到达服务器。

In [ ]:
def ask_user(server: str, tool: str, args: dict) -> bool:
    decision = not tool.startswith("write")              # scripted user: allow reads, deny writes
    print(f"  [approval] {server}.{tool}({json.dumps(args)[:60]}) -> {'ALLOW' if decision else 'DENY'}")
    return decision


async def guarded_call(client: Client, server: str, tool: str, args: dict):
    if not ask_user(server, tool, args):
        return {"isError": True, "content": "The user declined this tool call."}
    r = await client.call_tool(tool, args)
    return {"isError": r.is_error, "content": r.structured_content or r.content[0].text}


async with Client(params_for(cfg["mcpServers"]["notes"])) as c:
    print(await guarded_call(c, "notes", "read_file", {"path": str(work / "Desktop" / "todo.md")}))
    print(await guarded_call(c, "notes", "write_file", {"path": str(work / "Desktop" / "poem.txt"), "text": "roses"}))
print("poem.txt exists?", (work / "Desktop" / "poem.txt").exists())

### 6. 故障排查：复现典型故障
下面每个案例都是现实中会犯的错误。请对比<strong>宿主</strong>看到的内容（往往只是一个被关闭的连接）和<strong>服务器日志</strong>中的内容。正是这种差距，让文档首先把你引向 `mcp-server-NAME.log`（服务器日志文件）。

In [ ]:
def root_cause(exc: BaseException) -> BaseException:
    while isinstance(exc, BaseExceptionGroup) and len(exc.exceptions) == 1:
        exc = exc.exceptions[0]
    return exc


good = cfg["mcpServers"]["notes"]
CASES = {
    "relative path": {**good, "args": [str(SERVER), "Desktop"]},
    "missing env var": {k: v for k, v in good.items() if k != "env"},
    "non-existent command": {**good, "command": "pyhton3"},
}
for label, entry in CASES.items():
    log_file = LOGS / f"case-{label.replace(' ', '-')}.log"
    t0 = time.time()
    try:
        with open(log_file, "w") as errlog:
            async with Client(stdio_client(params_for(entry), errlog=errlog)) as c:
                await c.list_tools()
        outcome = "connected?!"
    except BaseException as exc:
        cause = root_cause(exc)
        outcome = f"{type(cause).__name__}: {cause}"
    print(f"### {label}  ({time.time() - t0:.1f}s)\n  host sees : {outcome}")
    print(f"  server log: {log_file.read_text().strip() or '(empty: the process never started)'}\n")

这些修复方法与文档的检查清单一致：使用<strong>绝对路径</strong>（服务器把 `Desktop` 相对于宿主的工作目录（Working Directory）进行了解析），把必需的密钥放进 <strong>`env`</strong>，并让 `command` 是宿主能找到的程序。先手动运行一遍，或者给出绝对路径，例如 `which uv` 的输出。

### 7. 一个基于 Streamable HTTP、使用 bearer 令牌的“远程”服务器
远程服务器（Remote Server）通过 <strong>URL</strong> 而不是启动命令来访问。在 Claude 中，这就是自定义连接器（Custom Connector），客户端会在你连接时完成认证（通常是 OAuth）。这里用一个 localhost 服务器来代替互联网。一个小小的 ASGI 包装层会拒绝所有不带正确 `Authorization` 请求头的请求，返回 `401` 和一个 `WWW-Authenticate` 质询，这正是受保护的 MCP 服务器表明需要认证的方式（模块 6 会讲真正的 OAuth 流程）。

In [ ]:
import httpx2
import uvicorn
from mcp.client.streamable_http import streamable_http_client
from mcp.server import MCPServer

remote = MCPServer("tickets")


@remote.tool()
def open_tickets(team: str) -> list[str]:
    """Open tickets for a team."""
    return {"platform": ["PLAT-7 flaky deploy", "PLAT-9 disk alert"]}.get(team, [])


TOKEN = "demo-access-token"
requests_seen = []
mcp_app = remote.streamable_http_app()


async def protected(scope, receive, send):
    if scope["type"] == "http":
        headers = dict(scope["headers"])
        auth = headers.get(b"authorization", b"").decode()
        requests_seen.append((headers.get(b"mcp-method", b"-").decode(), "token" if auth else "no token"))
        if auth != f"Bearer {TOKEN}":
            await send({"type": "http.response.start", "status": 401,
                        "headers": [(b"content-type", b"application/json"),
                                    (b"www-authenticate", b'Bearer realm="tickets"')]})
            await send({"type": "http.response.body", "body": b'{"error": "invalid_token"}'})
            return
    await mcp_app(scope, receive, send)


sock = socket.socket()
sock.bind(("127.0.0.1", 0))
PORT = sock.getsockname()[1]
sock.close()
http_server = uvicorn.Server(uvicorn.Config(protected, host="127.0.0.1", port=PORT, log_level="warning"))
threading.Thread(target=http_server.run, daemon=True).start()
while not http_server.started:
    await asyncio.sleep(0.05)
URL = f"http://127.0.0.1:{PORT}/mcp"
print("remote server at", URL)

通过 URL 连接。要发送自定义请求头，就给 `streamable_http_client` 传入你自己的 `httpx2.AsyncClient`（SDK 使用的 HTTP 库），再把这个传输（Transport）传给 `Client`。

In [ ]:
async with httpx2.AsyncClient(headers={"Authorization": f"Bearer {TOKEN}"}, trust_env=False) as http:
    async with Client(streamable_http_client(URL, http_client=http)) as c:
        print("negotiated:", c.protocol_version)
        r = await c.call_tool("open_tickets", {"team": "platform"})
        print("open_tickets ->", r.structured_content)

try:
    async with Client(URL) as c:                       # no Authorization header
        await c.list_tools()
except BaseException as exc:
    print("\nwithout a token ->", type(root_cause(exc)).__name__, root_cause(exc))

async with httpx2.AsyncClient(trust_env=False) as raw:
    resp = await raw.post(URL, json={"jsonrpc": "2.0", "id": 1, "method": "server/discover", "params": {}},
                          headers={"Accept": "application/json, text/event-stream",
                                   "MCP-Protocol-Version": "2026-07-28", "Mcp-Method": "server/discover"})
print("raw HTTP:", resp.status_code, "| WWW-Authenticate:", resp.headers.get("www-authenticate"), "|", resp.text)
print("\nwhat the server saw (Mcp-Method, auth):", requests_seen)

SDK 只把未经认证的尝试报告为一个笼统的错误。真正告诉你这是认证问题的，是 <strong>HTTP 状态码和 `WWW-Authenticate` 请求头</strong>，这就是宿主日志（或 Inspector）之所以重要的原因。服务器日志还显示，客户端先探测了 `server/discover`，然后在放弃之前又重试了旧版握手（`-`，没有 `Mcp-Method`）。

### 8. 一份配置，本地与远程兼顾
许多工具复用了 `mcpServers` 的结构，并增加了一种远程条目类型。例如 MCP 检查器（MCP Inspector）的服务器目录文件（Catalog File）接受带有 `url` 和 `headers` 的 `"type": "http"`。Claude Desktop 自己的 `claude_desktop_config.json` 在文档中只用于<strong>本地</strong>启动条目；远程服务器在 Claude 中是通过 Settings → Connectors 以<strong>自定义连接器</strong>的形式添加的。下面是一个宿主函数，可以从这样的目录中连接任意一种服务器。

In [ ]:
catalog = {"mcpServers": {
    "notes": cfg["mcpServers"]["notes"],
    "tickets": {"type": "http", "url": URL, "headers": {"Authorization": f"Bearer {TOKEN}"}},
}}


async def open_entry(stack: AsyncExitStack, entry: dict) -> Client:
    if entry.get("type") == "http":
        http = await stack.enter_async_context(httpx2.AsyncClient(headers=entry.get("headers"), trust_env=False))
        return await stack.enter_async_context(Client(streamable_http_client(entry["url"], http_client=http)))
    return await stack.enter_async_context(Client(params_for(entry)))


async with AsyncExitStack() as stack:
    for name, entry in catalog["mcpServers"].items():
        c = await open_entry(stack, entry)
        kind = "remote (Streamable HTTP)" if entry.get("type") == "http" else "local (stdio)"
        print(f"{name:<8} {kind:<26} tools={[t.name for t in (await c.list_tools()).tools]}")

http_server.should_exit = True

### 9. 可选：让 Claude 使用已配置的服务器
设置了 `ANTHROPIC_API_KEY` 之后，这就是一个微型的 Claude Desktop：汇集所有已配置服务器的工具（Tool）（以 `server__tool` 形式加命名空间），使用第 5 节的同意关卡，以及第 5.2 课中的智能体循环（Agent Loop）。

In [ ]:
if os.environ.get("ANTHROPIC_API_KEY"):
    import anthropic

    llm = anthropic.Anthropic()
    async with AsyncExitStack() as stack:
        clients = await connect_all(stack, cfg)
        tools, route = [], {}
        for server_name, client in clients.items():
            for t in (await client.list_tools()).tools:
                tools.append({"name": f"{server_name}__{t.name}", "description": t.description or "",
                              "input_schema": t.input_schema})
                route[f"{server_name}__{t.name}"] = (server_name, client, t.name)
        messages = [{"role": "user", "content": "What's on my todo list? Then save a two-line poem about it "
                     "to my Desktop as poem.txt."}]
        response = llm.messages.create(model="claude-opus-5-5", max_tokens=16000, tools=tools, messages=messages)
        while response.stop_reason == "tool_use":
            messages.append({"role": "assistant", "content": response.content})
            results = []
            for block in response.content:
                if block.type == "tool_use":
                    server_name, client, tool = route[block.name]
                    out = await guarded_call(client, server_name, tool, block.input)
                    results.append({"type": "tool_result", "tool_use_id": block.id,
                                    "content": json.dumps(out["content"]), "is_error": out["isError"]})
            messages.append({"role": "user", "content": results})
            response = llm.messages.create(model="claude-opus-5-5", max_tokens=16000, tools=tools,
                                           messages=messages)
        print("Claude:", "".join(b.text for b in response.content if b.type == "text"))
else:
    print("skipped: set ANTHROPIC_API_KEY to watch Claude read your todo list and get its write request denied.")

### 动手练习
1. 在 `config` 中添加第二个服务器（例如一份只提供 `Downloads` 的 `notes_server.py` 副本）。重新运行第 3 节：两个服务器是否都被列出？每个是否都有自己的 `mcp-server-NAME.log`？
2. 在第 6 节的“relative path”案例中加入 `"cwd": str(work)`。现在能启动了吗？为什么文档仍然建议使用绝对路径，而不是依赖 `cwd`？
3. 修改远程包装层，让它也接受 `X-API-Key` 请求头，并用它来连接。当服务器使用静态 API 密钥而不是 OAuth 时，宿主会失去什么（想想权限范围、过期和吊销）？
4. 让 `ask_user` 在第一次批准后记住“对这个服务器始终允许 `read_file`”。这种授权应该存储在哪里？又应该在什么时候过期？